# Reachy Mini Simulation

## 注意

このノートブックはColab課金ユーザのみ実行可能です。

無課金ユーザはアカウント停止の原因になります。

実行前に[利用規約](https://research.google.com/colaboratory/faq.html#disallowed-activities)を確認してください。

## 事前準備

### SSHキー

ローカルPCでSSHキーを作成

```sh
# ターミナルで実行
ssh-keygen -t ed25519 -C "your_email@example.com"
# または RSA の場合
ssh-keygen -t rsa -b 4096 -C "your_email@example.com"

# Enter キーを3回押す（デフォルト設定の場合）
# パスフレーズを設定することも可能
```

公開鍵の確認

```sh
cat ~/.ssh/id_ed25519.pub
# または
cat ~/.ssh/id_rsa.pub
出力例:

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAI... your_email@example.com
```

取得した公開鍵は、Google ColabのKeyにPUBLIC_KEYとして設定する。

### GitHub Personal Access Token（オプション）

[PAT](https://github.com/settings/personal-access-tokens)に行き、Generate new tokenを作成する。

特定のリポジトリに対して、「Contents」パーミッションのみを付与する。

取得したPATは、Google ColabのKeyにGITHUB_TOKENとして設定する。

## SSHサーバの起動

In [ ]:
# 公開鍵を取得
from google.colab import userdata
PUBLIC_KEY = userdata.get('PUBLIC_KEY')
assert PUBLIC_KEY

# SSH ディレクトリの作成
ssh_dir = "/root/.ssh"
!mkdir -p {ssh_dir}
!chmod 700 {ssh_dir}

# authorized_keys に書き込み
authorized_keys_path = f"{ssh_dir}/authorized_keys"
with open(authorized_keys_path, 'w') as f:
    f.write(PUBLIC_KEY.strip())

# パーミッション設定
!chmod 600 {authorized_keys_path}

# SSHサーバーのインストールと設定
print("📦 SSH サーバーをインストールしています...")
!apt-get update -qq
!apt-get install -y openssh-server > /dev/null 2>&1

# SSH設定ファイルを作成（rootログイン許可）
sshd_config = """Port 22
PermitRootLogin prohibit-password
PubkeyAuthentication yes
PasswordAuthentication no
ChallengeResponseAuthentication no
UsePAM yes
X11Forwarding yes
PrintMotd no
AcceptEnv LANG LC_*
Subsystem sftp /usr/lib/openssh/sftp-server
"""

with open('/etc/ssh/sshd_config', 'w') as f:
    f.write(sshd_config)

# SSHサービスを起動
!service ssh start
!service ssh status | grep Active

print("\n✅ SSH サーバーが起動しました")
print("   - ルートログイン: 公開鍵認証のみ許可")
print("   - パスワード認証: 無効")
print("   - 公開鍵認証: 有効\n")

# ユーザー情報を表示
print(f"📋 ユーザー情報:")
!id root

## 環境構築

In [ ]:
# tmux設定ファイルを作成
tmux_conf = r'''
# ------------------------------
# 基本設定
# ------------------------------

# マウスを有効化
set -g mouse on

# ペインの自動リネームを無効化
set -g automatic-rename off
set -g allow-rename off

# ペイン番号を1から開始（デフォルトは0）
setw -g pane-base-index 1

# ------------------------------
# キーバインド
# ------------------------------

# ペインの名前を変更するショートカット
bind t command-prompt -p "ペイン名:" "select-pane -T '%%'"

# ペインの同期（synchronize-panes）のトグル
bind S setw synchronize-panes \; display "同期: #{?synchronize-panes,ON,OFF}"

# ペインを水平方向に均等に分割
bind H select-layout even-horizontal

# ペインを垂直方向に均等に分割
bind V select-layout even-vertical

# ------------------------------
# ペインタイトルの表示
# ------------------------------

# ペインボーダーにタイトルを表示
set -g pane-border-status top
set -g pane-border-format "#{pane_index}: #{pane_title}"

# ------------------------------
# 便利な設定
# ------------------------------

# エスケープキーの遅延を解消
set -s escape-time 0

# ヒストリの上限を増やす
set -g history-limit 10000

# ウィンドウ番号を1から開始（デフォルトは0）
set -g base-index 1

# ウィンドウを閉じた時に番号を詰める
set -g renumber-windows on

# ------------------------------
# キーバインド
# ------------------------------

# 設定ファイルのリロード
bind r source-file ~/.tmux.conf \; display "設定をリロードしました"

# ペインの分割（より直感的なキー）
bind | split-window -h -c "#{pane_current_path}"
bind - split-window -v -c "#{pane_current_path}"

# ペイン間の移動（Vim風）
bind h select-pane -L
bind j select-pane -D
bind k select-pane -U
bind l select-pane -R

# コピーモードでVim風のキーバインドを使用
setw -g mode-keys vi
bind -T copy-mode-vi v send -X begin-selection
bind -T copy-mode-vi y send -X copy-selection-and-cancel

# ------------------------------
# ビジュアル設定
# ------------------------------

# 256色対応
set -g default-terminal "tmux-256color"

# ステータスバーの色とフォーマット
set -g status-style 'bg=colour234 fg=colour137'
set -g status-right '#[fg=colour233,bg=colour241,bold] %Y/%m/%d #[fg=colour233,bg=colour245,bold] %H:%M '

# アクティブなペインのボーダー色
set -g pane-border-style 'fg=colour238'
set -g pane-active-border-style 'fg=colour51'

# ステータスバーの更新間隔（秒）
set -g status-interval 1

# True Color サポート
set -ga terminal-features ",*:RGB"

# ペインの番号表示を見やすく
set -g display-panes-colour colour166
set -g display-panes-active-colour colour33

# メッセージの色
set -g message-style 'fg=colour232 bg=colour166 bold'

# プレフィックスキーを Ctrl+a に変更
unbind C-b
set -g prefix C-a
bind C-a send-prefix

# アクティブなウィンドウのステータス表示をカスタマイズ
setw -g window-status-current-style 'fg=colour1 bg=colour238 bold'
setw -g window-status-current-format ' #I#[fg=colour250]:#[fg=colour255]#W#[fg=colour50]#{?window_zoomed_flag, 🔍,} '

# 非アクティブなウィンドウのステータス表示
setw -g window-status-style 'fg=colour138 bg=colour235'
setw -g window-status-format ' #I#[fg=colour237]:#[fg=colour250]#W#[fg=colour244]#F '
'''

with open('/root/.tmux.conf', 'w') as f:
    f.write(tmux_conf)
print("✅ tmux設定ファイルを作成しました。")

In [ ]:
import subprocess

# .bashrcに cd /content を追加
!echo 'cd /content' >> /root/.bashrc

# Google Colab 環境変数を.bashrcに追加（PYTHONPATHを含む）
!env | grep -E "^(LD_LIBRARY_PATH|PATH|PYTHONPATH|CUDA|NVIDIA_VISIBLE|COLAB_GPU)=" | \
    sed 's/^/export /' >> /root/.bashrc

print("✅ 環境変数が /root/.bashrc に追加されました。\n")

# 先にbunをインストール（並列インストールで使用するため）
print("📦 bunをインストールしています...")
!curl -fsSL https://bun.sh/install | bash

# 並列インストールコマンド（npmパッケージはbunでまとめてインストール）
commands = [
    "curl -fsSL https://claude.ai/install.sh | bash",
    "$HOME/.bun/bin/bun add -g @openai/codex @google/gemini-cli @github/copilot",
    "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y",
]

# 並列実行
processes = [subprocess.Popen(cmd, shell=True) for cmd in commands]

# 全て完了するまで待機
print("⏳ ツールをインストールしています。完了までお待ちください...\n")
for p in processes:
    p.wait()

# .bashrc設定
with open('/root/.bashrc', 'a') as f:
    f.write('\n# ツールのPATH\n')
    f.write('export PATH="$HOME/.bun/bin:$HOME/.local/bin:$HOME/.claude/bin:$HOME/.cargo/bin:$PATH"\n')

print("\n✅ すべてのツールがインストールされました。")

## Cloudflaredの起動

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /tmp/cloudflared
!chmod +x /tmp/cloudflared
!/tmp/cloudflared tunnel --url ssh://localhost:22

トンネルURLが「https://comparing-trim-background-improve.trycloudflare.com」の場合、以下をローカルPCで設定する。

~/.ssh/config に追加:

```sh
Host colab
HostName comparing-trim-background-improve.trycloudflare.com
User root
ProxyCommand cloudflared access ssh --hostname %h
IdentityFile ~/.ssh/id_ed25519
IdentitiesOnly yes
StrictHostKeyChecking no
UserKnownHostsFile /dev/null
ServerAliveInterval 60
ServerAliveCountMax 3
```

これにより `ssh colab` やVSCode・AntigravityでSSH接続できる。

GitHubとの連携は、以下のコマンドを実行する。アクセス制御したい場合は[トークンを発行](https://github.com/settings/personal-access-tokens)する。

`gh auth login`